### 01. Descarga de datos

Esta notebook intenta generar procesos, funciones para la descarga de los productos satelitales como:
- AOD,
- Variables metereologicas
- Uso de suelo
- Elevacion
- Composicion de aerosoles
- Otras
Para luego poder ser utilizadas en la prediccion del PM2.5

In [33]:
# Library
#import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np
import scipy.stats
from sklearn.metrics import mean_squared_error
from math import sqrt
import os
from os import listdir
from datetime import datetime
import pandas as pd
from matplotlib.dates import DateFormatter
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
import requests
from pathlib import Path
import os
from dotenv import load_dotenv
import sys
from config import SITES
from dateutil.relativedelta import relativedelta
import cdsapi
print("librerias ok")

librerias ok


In [4]:
#Configuracion de las variables de entorno
load_dotenv()
TOKEN_MAIAC = os.getenv("TOKEN_MAIAC")
#print(TOKEN_MAIAC)

In [ ]:
# #Setear las rutas
# # Ruta raíz del proyecto
# ROOT_DIR = Path(__file__).resolve().parent.parent

# # Agregar raíz al PATH de Python
# sys.path.append(str(ROOT_DIR))

# # Carpetas del proyecto
# DATA_RAW_DIR = ROOT_DIR / "data/raw"
# MAIAC_DIR = DATA_RAW_DIR / "MAIAC"


# ST
# -71.0653675613302,-33.07880767505593, -70.3061109228608,-33.74283779348805

In [46]:
# Directorio donde está el notebook
NOTEBOOK_DIR = Path.cwd()

# Subir un nivel → raíz del proyecto
ROOT_DIR = NOTEBOOK_DIR.parent.parent

# Carpetas del proyecto
DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
AOD_DIR = DATA_RAW_DIR / "AOD"
NDVI_DIR = DATA_RAW_DIR / "NDVI"
ERA5_DIR = DATA_RAW_DIR / "ERA5"
print("Notebook:", NOTEBOOK_DIR)
print("Root:", ROOT_DIR)
print("AOD:", AOD_DIR)
print("NDVI:", NDVI_DIR)
print("ERA5:", ERA5_DIR)

Notebook: d:\Josefina\Proyectos\Tesis\code_py\Notebooks\00-Descarga-datos
Root: d:\Josefina\Proyectos\Tesis\code_py
AOD: d:\Josefina\Proyectos\Tesis\code_py\data\raw\AOD
NDVI: d:\Josefina\Proyectos\Tesis\code_py\data\raw\NDVI
ERA5: d:\Josefina\Proyectos\Tesis\code_py\data\raw\ERA5


In [5]:
# Configuracion del sitio
PRODUCT = "MCD19A2"
DATE = "2026-08-13"

SITES = {
    "San Pablo": {
        "west": -47.18761699953164,
        "south": -23.769825082026546,
        "east": -46.2989251168905,
        "north": -23.15380334079614
    },

    "Santiago": {
        "west": -71.0653675613302, #supizq
        "south": -33.74283779348805, #AbajDer
        "east": -70.3061109228608, #abajder
        "north": -33.07880767505593 #SupIzq
    },

"Medellin": {
        "west": -75.73349373413268,
        "south": 6.064656084590012,
        "east": -75.3546571948446,
        "north": 6.45881903338495
    },

"Mexico": {
        "west": -99.62981206846747,
        "south": 18.79096843909478,
        "east": -98.41583428026645,
        "north": 19.9977998136125
    },

}



In [6]:
def validate_bbox(bbox):

    if not -180 <= bbox["west"] <= 180:
        raise ValueError("La longitud west debe estar entre -180 y 180.")

    if not -180 <= bbox["east"] <= 180:
        raise ValueError("La longitud east debe estar entre -180 y 180.")

    if not -90 <= bbox["south"] <= 90:
        raise ValueError("La latitud south debe estar entre -90 y 90.")

    if not -90 <= bbox["north"] <= 90:
        raise ValueError("La latitud north debe estar entre -90 y 90.")

    if bbox["west"] >= bbox["east"]:
        raise ValueError("West debe ser menor que East.")

    if bbox["south"] >= bbox["north"]:
        raise ValueError("South debe ser menor que North.")

In [ ]:
# DESCARGA AOD
def aod_download (site, date):

    # Configuracion
    SITE = site
    OUTPUT_DIR = MAIAC_DIR
    url = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"
    headers = {"Authorization": f"Bearer {TOKEN_MAIAC}"}

    #Fecha de interes
    date_range = f"{date}..{date}"
    #Coordenadas
    if site not in SITES:
        raise ValueError(
            f"Sitio '{site}' no configurado. "
            f"Sitios disponibles: {list(SITES.keys())}"
        )

    BBOX = SITES[site]
    # Convertir BBOX al formato requerido por LAADS
    bbox_lads = (
        f"[BBOX]"
        f"W{BBOX['west']} "
        f"S{BBOX['south']} "
        f"E{BBOX['east']} "
        f"N{BBOX['north']}"
    )

    # Parametros para la API
    params = {
        "products": "MCD19A2",
        "temporalRanges": date_range,
        "regions": bbox_lads,
        "formats": "json"
    }

    # Crear carpeta de descarga
    #OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Buscar archivos
    response = requests.get(
        url,
        headers = headers,
        params = params
    )

    response.raise_for_status()
    data = response.json()

    # Validar disponibilidad de datos 
    if data["file_count"] == 0:
        raise RuntimeError(
            f"No se encontraron archivos de AOD "
            f"para el sitio '{site}' en la fecha {date}.")
    print(f"Archivos encontrados: {data['file_count']}")


    # Descargar archivos encontrados
    downloaded = 0
    skipped = 0
    for archivo in data["content"]:
        nombre = archivo["name"]
        download_url = archivo["downloadsLink"]
        output_file = OUTPUT_DIR / nombre
        print(f"Descargando: {nombre}")
        # Si ya existe, no lo volvemos a descargar
        if output_file.exists():
            print("El archivo ya existe. Se omite.")
            skipped += 1
            continue
        with requests.get(
            download_url,
            headers = headers,
            stream = True
        ) as r:
            
            r.raise_for_status()
            with open(output_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        downloaded += 1

        print("Descarga completa")

    return {"site": site, "date": date, "files_found": data["file_count"], "files_downloaded": downloaded, "files_skipped": skipped}


In [29]:
# NDVI MOD13A3

def ndvi_download(site, end_date):

# Configuracion
    OUTPUT_DIR = NDVI_DIR
    url = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"
    headers = {
        "Authorization": f"Bearer {TOKEN_MAIAC}"
    }

    # Validar sitio
    if site not in SITES:
        raise ValueError(
            f"Sitio '{site}' no configurado. "
            f"Sitios disponibles: {list(SITES.keys())}"
        )
    # BBox
    BBOX = SITES[site]
    bbox_lads = (
        f"[BBOX]"
        f"W{BBOX['west']} "
        f"S{BBOX['south']} "
        f"E{BBOX['east']} "
        f"N{BBOX['north']}"
    )

    # Fecha
    end_date = datetime.strptime(
        end_date,
        "%Y-%m-%d"
    )
    search_date = end_date.replace(day=1)

    # Buscar ultimo dato disponible
    while True:
        test_date = search_date.strftime("%Y-%m-%d")
        date_range = f"{test_date}..{test_date}"
        params = {
            "products": "MOD13A3",
            "temporalRanges": date_range,
            "regions": bbox_lads,
            "formats": "json"
        }
        response = requests.get(
            url,
            headers=headers,
            params=params
        )
        response.raise_for_status()
        data = response.json()
        archivos = data.get("content", [])

        print(f"Buscando MOD13A3: {test_date} → {len(archivos)} archivos")

        if archivos:
            print(f"Último MOD13A3 disponible: {test_date}")
            break
        # Retroceder un mes
        search_date -= relativedelta(months=1)

    # Descargar el ultimo dato disponible
    downloaded = 0
    skipped = 0

    for archivo in archivos:
        nombre = archivo["name"]
        download_url = archivo["downloadsLink"]
        output_file = OUTPUT_DIR / nombre
        print(f"Descargando: {nombre}")
        if output_file.exists():
            print("El archivo ya existe. Se omite.")
            skipped += 1
            continue
        with requests.get(
            download_url,
            headers = headers,
            stream = True
        ) as r:
            r.raise_for_status()
            with open(output_file, "wb") as f:
                for chunk in r.iter_content(
                    chunk_size=8192
                ):
                    if chunk:
                        f.write(chunk)
        downloaded += 1
        print("Descarga completa")

    # Se muestra un resumen de la info descargada

    return {"site": site,
        "date": test_date,
        "files_found": len(archivos),
        "files_downloaded": downloaded,
        "files_skipped": skipped
    }

In [73]:
site = "Santiago"
date = "2026-08-10"
def era_download(site, date):

    dataset = "reanalysis-era5-land"
    #dataset ="reanalysis-era5-single-levels"
    # Fecha
    date_dt = datetime.strptime(date, "%Y-%m-%d")
    year = date_dt.year
    month = date_dt.month
    day = date_dt.day

    # Validar sitio
    if site not in SITES:
        raise ValueError(
            f"Sitio '{site}' no configurado. "
            f"Sitios disponibles: {list(SITES.keys())}"
        )
    BBOX = SITES[site]
    bbox_era = [BBOX['north'],BBOX['west'],BBOX['south'],BBOX['east']]


    #Archivo salida
    output_file = ERA5_DIR / f"ERA5_{date}.nc"

    if output_file.exists():

        print(f"El archivo ya existe: {output_file.name}")

        return {
            "site": site,
            "date": date,
            "file": output_file.name,
            "downloaded": False,
            "skipped": True
        }


    request = {
        "variable": [
            "10m_u_component_of_wind",
            # "10m_v_component_of_wind",
            # "2m_dewpoint_temperature",
            # "2m_temperature",
            # "surface_pressure",
            # "total_precipitation",
            # "boundary_layer_height"
        ],
        "year": year,
        "month": month,
        "day": day,
        "time": [
            "00:00", "01:00"#, "02:00",
            # "03:00", "04:00", "05:00",
            # "06:00", "07:00", "08:00",
            # "09:00", "10:00", "11:00",
            # "12:00", "13:00", "14:00",
            # "15:00", "16:00", "17:00",
            # "18:00", "19:00", "20:00",
            # "21:00", "22:00", "23:00"
        ],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": bbox_era
    }
    archivos = request.get("content", [])
    client = cdsapi.Client()
    
    client.retrieve(dataset, request, target= output_file)#.download()

    print(f"Descarga completa:{output_file.name}")
    # Se muestra un resumen de la info descargada
    return {
        "site": site,
        "date": date,
        "file": output_file.name,
        "downloaded": True,
        "skipped": False }


### Testeos de funciones

In [ ]:
# AOD
aod_download (site = "Medellin", date = "2026-07-13")

Archivos encontrados: 1
Descargando: MCD19A2.A2026194.h10v08.061.2026198175716.hdf
El archivo ya existe. Se omite.


{'site': 'Medellin',
 'date': '2026-07-13',
 'files_found': 1,
 'files_downloaded': 0,
 'files_skipped': 1}

In [31]:
# NDVI
ndvi_download(site="Santiago", end_date="2026-08-17")

Buscando MOD13A3: 2026-08-01 → 0 archivos
Buscando MOD13A3: 2026-07-01 → 2 archivos
Último MOD13A3 disponible: 2026-07-01
Descargando: MOD13A3.A2026182.h11v12.061.2026229001400.hdf
Descarga completa
Descargando: MOD13A3.A2026182.h12v12.061.2026229001319.hdf
Descarga completa


{'site': 'Santiago',
 'date': '2026-07-01',
 'files_found': 2,
 'files_downloaded': 2,
 'files_skipped': 0}

In [77]:
# ERA 5
era_download(site = "Santiago", date = "2026-08-10")

2026-08-17 13:09:33,120 INFO Request ID is 0a7f0a29-9dd9-4a12-af89-697be66df079
2026-08-17 13:09:33,744 INFO status has been updated to accepted
2026-08-17 13:10:17,461 INFO status has been updated to running
2026-08-17 13:10:34,851 INFO status has been updated to successful
                                                                                         

Descarga completa:ERA5_2026-08-10.nc


{'site': 'Santiago',
 'date': '2026-08-10',
 'file': 'ERA5_2026-08-10.nc',
 'downloaded': True,
 'skipped': False}